# Invoice Information Extraction — Donut Fine-tuning
Run cells top to bottom. Runtime: **Runtime > Change runtime type > T4 GPU**.

**Model choice: `naver-clova-ix/donut-base`.**
Donut ("Document Understanding Transformer") is an OCR-free image-to-JSON document
transformer (Swin encoder + BART-style decoder). It's a strong fit here because:
1. The task IS "image -> structured JSON" — exactly what Donut was pretrained/designed for
   (validated on CORD, a receipt-parsing task structurally identical to this invoice task).
2. It needs no separate OCR + layout step at inference time (OCR-free), keeping the
   pipeline in the diagram simple: Image -> Preprocessing -> Model -> JSON.
3. It fine-tunes well from small datasets (hundreds of examples) in the CORD/RVL-CDIP
   literature, realistic for our ~340 training images in a one-day budget.
Alternative considered: LayoutLMv3 (token classification over OCR+bounding boxes) — more
accurate for pure key-value tasks but needs a separate OCR/box pipeline and doesn't
natively handle repeating line-item structures as cleanly as Donut's sequence generation.


In [ ]:
!pip install -q "transformers>=4.41" datasets sentencepiece accelerate evaluate


## 1. Mount Drive and unzip your project data
Before this: upload `invoice_project.zip` (contains `schema.py` and `data/splits/`,
produced by the local steps 01-03) to your Google Drive, e.g. `MyDrive/invoice_project.zip`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
ZIP_PATH = "/content/drive/MyDrive/invoice_project.zip"   # <-- change if you placed it elsewhere
EXTRACT_DIR = "/content/invoice_project"
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print(os.listdir(EXTRACT_DIR))


In [ ]:
import sys, os

def find_file(root, filename):
    """Search anywhere under root for filename (handles zip tools that
    flatten or rename intermediate folders differently across OSes)."""
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return dirpath
    return None

schema_dir = find_file(EXTRACT_DIR, "schema.py")
if schema_dir is None:
    raise FileNotFoundError(
        f"Could not find schema.py anywhere under {EXTRACT_DIR}. "
        f"Check that invoice_project.zip actually contains schema.py, and that ZIP_PATH above is correct."
    )
sys.path.insert(0, schema_dir)
import schema
print(f"Found schema.py in: {schema_dir}")
print(len(schema.ALL_SPECIAL_TAGS), "special tags to add, e.g.:", schema.ALL_SPECIAL_TAGS[:5])


## 2. Load Donut processor + model, add our task's special tokens

In [ ]:
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel, VisionEncoderDecoderConfig

MODEL_NAME = "naver-clova-ix/donut-base"
MAX_LENGTH = 768          # generous for our nested schema incl. multiple line items
IMAGE_SIZE = [960, 1280]  # (width, height) resize target — donut-base default resolution

config = VisionEncoderDecoderConfig.from_pretrained(MODEL_NAME)
config.encoder.image_size = IMAGE_SIZE
config.decoder.max_length = MAX_LENGTH

processor = DonutProcessor.from_pretrained(MODEL_NAME)
processor.image_processor.size = {"height": IMAGE_SIZE[1], "width": IMAGE_SIZE[0]}
processor.image_processor.do_align_long_axis = False

model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME, config=config)

# add our schema's tags as special tokens so they're single tokens, not sub-word soup
new_tokens = schema.ALL_SPECIAL_TAGS
num_added = processor.tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
model.decoder.resize_token_embeddings(len(processor.tokenizer))
print(f"Added {num_added} special tokens. Vocab size now {len(processor.tokenizer)}")

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids([schema.TASK_START_TOKEN])[0]
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.vocab_size = len(processor.tokenizer)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("device:", device)


## 3. Dataset
Reads `manifest.jsonl` ({file_name, target}) + the split's images/ folder.
Image preprocessing: Donut's own image processor (resize to 960x1280, rescale to
[0,1], normalize with ImageNet mean/std) — no separate preprocessing step needed,
it's built into `processor(image, ...)`.


In [ ]:
import json, os
from PIL import Image
from torch.utils.data import Dataset

class InvoiceDonutDataset(Dataset):
    def __init__(self, split_dir, processor, max_length=MAX_LENGTH):
        self.split_dir = split_dir
        self.processor = processor
        self.max_length = max_length
        self.examples = []
        with open(os.path.join(split_dir, "manifest.jsonl")) as f:
            for line in f:
                self.examples.append(json.loads(line))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        image = Image.open(os.path.join(self.split_dir, "images", ex["file_name"])).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            ex["target"], add_special_tokens=False, max_length=self.max_length,
            padding="max_length", truncation=True, return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100  # ignored by loss
        return {"pixel_values": pixel_values, "labels": labels}

def find_data_root(root):
    """Search anywhere under root for a folder that has train/val/test
    subfolders each containing manifest.jsonl -- works no matter whether
    your zip tool preserved 'data/splits/...' or flattened it to just
    'splits/...' (a known Windows Compress-Archive quirk)."""
    for dirpath, dirnames, _ in os.walk(root):
        if all(os.path.isfile(os.path.join(dirpath, d, "manifest.jsonl")) for d in ("train", "val", "test")
               if d in dirnames) and {"train", "val", "test"}.issubset(set(dirnames)):
            return dirpath
    return None

DATA_ROOT = find_data_root(EXTRACT_DIR)
if DATA_ROOT is None:
    raise FileNotFoundError(
        f"Could not find train/val/test folders (each with manifest.jsonl) anywhere under {EXTRACT_DIR}. "
        f"Run: import subprocess; print(subprocess.run(['find', EXTRACT_DIR], capture_output=True, text=True).stdout) "
        f"to see what actually got extracted, and check invoice_project.zip's contents."
    )
print(f"Found dataset splits in: {DATA_ROOT}")
train_ds = InvoiceDonutDataset(os.path.join(DATA_ROOT, "train"), processor)
val_ds = InvoiceDonutDataset(os.path.join(DATA_ROOT, "val"), processor)
test_ds = InvoiceDonutDataset(os.path.join(DATA_ROOT, "test"), processor)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")


## 4. Smoke test (dataset -> preprocessing -> model -> loss -> backward -> optimizer step)
Run this BEFORE the full training run. It must finish with a finite, non-NaN loss.


In [ ]:
from torch.utils.data import DataLoader

smoke_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
batch = next(iter(smoke_loader))
pixel_values = batch["pixel_values"].to(device)
labels = batch["labels"].to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
model.train()
outputs = model(pixel_values=pixel_values, labels=labels)
loss = outputs.loss
print("smoke-test loss (should be a finite number, roughly ln(vocab_size) ~ 10-11 before training):", loss.item())
assert torch.isfinite(loss), "Loss is NaN/Inf -- stop and debug before full training"

loss.backward()
optimizer.step()
optimizer.zero_grad()
print("SMOKE TEST PASSED: forward, backward, and optimizer step all completed cleanly.")


## 5. Full training run

**Hyperparameters (also documented in README.md):**
- Learning rate: 3e-5
- Batch size: 2 per device, gradient accumulation 4 (effective batch size 8)
- Epochs: 30 (small dataset; Donut fine-tuning on ~300-400 examples typically needs
  many epochs to converge — monitored via val loss / early best-checkpoint selection)
- Optimizer: AdamW (`adamw_torch`)
- Loss function: token-level cross-entropy (built into `VisionEncoderDecoderModel`'s
  decoder — this is what `outputs.loss` above already was)
- Image preprocessing: resize to 960x1280, rescale + ImageNet normalization (Donut's
  own `DonutImageProcessor`, configured in step 2)

Model selection uses **validation loss only** — the test split is never touched until
the final evaluation notebook/script, to avoid test-set leakage into model selection.


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/invoice_donut_checkpoint"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=30,
    learning_rate=3e-5,
    optim="adamw_torch",
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    predict_with_generate=False,   # generation-based eval is done separately in 06_evaluation
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

train_result = trainer.train()
print(train_result)


## 6. Save the best checkpoint (by val loss) + processor, to Drive

In [ ]:
FINAL_DIR = "/content/drive/MyDrive/invoice_donut_final"
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
with open(os.path.join(FINAL_DIR, "training_config.json"), "w") as f:
    json.dump({
        "base_model": MODEL_NAME, "learning_rate": 3e-5, "batch_size_effective": 8,
        "per_device_train_batch_size": 2, "gradient_accumulation_steps": 4,
        "num_train_epochs": 30, "optimizer": "adamw_torch",
        "loss": "token-level cross-entropy (VisionEncoderDecoderModel decoder)",
        "image_size": IMAGE_SIZE, "max_length": MAX_LENGTH,
        "final_train_loss": train_result.training_loss,
    }, f, indent=2)
print("Saved to", FINAL_DIR)
print("Download this whole folder (or keep it on Drive) -- 05_inference and 06_evaluation load it by path.")
